# Processing Threshold Tuning

This notebook tests summary and connection thresholds against the processed Postgres data. The goal is to choose settings that preserve broad business-signal coverage while avoiding a connection graph that is dominated by weak or repetitive links.

The analysis is read-only. It does not update pipeline settings, summaries, connections, or source data.

## What This Tests

- Summary inputs: `relevance_threshold` and `initial_context_per_source`.
- Connection inputs: `similarity_threshold`, `connection_confidence_threshold`, and `max_connection_candidates_per_ticker`.
- Lookback: fixed at 90 days, matching the current processing contract for connections and ticker summaries.

The recommendation should favor enough coverage for every ticker before optimizing for precision. For this workflow, missing a useful lead is usually worse than including a few weaker items for analyst review.

In [3]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import subprocess
from dataclasses import dataclass

missing = [pkg for pkg in ["pandas", "psycopg"] if importlib.util.find_spec(pkg) is None]
if missing:
    raise ImportError(
        "Install notebook dependencies first: "
        "python -m pip install pandas psycopg[binary] matplotlib seaborn. "
        f"Missing: {', '.join(missing)}"
    )

import pandas as pd
import psycopg
from psycopg.rows import dict_row

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "eliza-ai-workbench-rg")
PROCESSING_FUNCTION_APP = os.getenv(
    "AZURE_PROCESSING_FUNCTION_APP", "eliza-ai-workbench-processing-func-18878"
)
POSTGRES_SERVER = os.getenv("AZURE_POSTGRES_SERVER", "eliza-ai-workbench-pg-58415")
CONNECT_TIMEOUT_SECONDS = 15
ADD_LOCAL_FIREWALL_RULE = False  # Set True only if local Postgres connections time out.
LOOKBACK_DAYS = 90
TICKERS = ["AMD", "SNDK", "FROG", "APP", "KVYO"]
SOURCES = ["reddit", "hacker_news", "github"]


def az_command() -> str:
    candidates = [
        "az",
        r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd",
    ]
    for candidate in candidates:
        try:
            subprocess.run([candidate, "--version"], check=True, capture_output=True, text=True)
            return candidate
        except Exception:
            continue
    raise RuntimeError("Azure CLI was not found. Set AZURE_POSTGRES_DSN in the notebook environment instead.")

## Connect To Postgres

The notebook first looks for `AZURE_POSTGRES_DSN` in the environment. If it is not set, it reads the setting from the Azure Processing Function App using the local Azure CLI login. If the setting is a Key Vault reference, the notebook resolves the secret value in memory and does not print it.

If the DSN loads but the first query times out, your local IP is not allowed by the Azure Postgres firewall. Set `ADD_LOCAL_FIREWALL_RULE = True` in the setup cell and rerun it to add a temporary `AllowLocalNotebook` rule for your current public IP.

In [4]:
def resolve_key_vault_reference(value: str, az: str) -> str:
    if not value.startswith("@Microsoft.KeyVault("):
        return value
    match = re.search(r"SecretUri=([^)]*)", value)
    if not match:
        raise RuntimeError("AZURE_POSTGRES_DSN is a Key Vault reference, but no SecretUri was found.")
    secret_uri = match.group(1)
    secret = subprocess.run(
        [az, "keyvault", "secret", "show", "--id", secret_uri, "--query", "value", "-o", "tsv"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if not secret:
        raise RuntimeError("Key Vault returned an empty AZURE_POSTGRES_DSN secret.")
    return secret


def current_public_ip() -> str:
    result = subprocess.run(
        ["powershell", "-NoProfile", "-Command", "(Invoke-RestMethod -Uri 'https://api.ipify.org').Trim()"],
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


def allow_current_ip_for_postgres() -> None:
    az = az_command()
    ip = current_public_ip()
    subprocess.run(
        [
            az,
            "postgres",
            "flexible-server",
            "firewall-rule",
            "create",
            "--resource-group",
            RESOURCE_GROUP,
            "--name",
            POSTGRES_SERVER,
            "--rule-name",
            "AllowLocalNotebook",
            "--start-ip-address",
            ip,
            "--end-ip-address",
            ip,
        ],
        check=True,
    )
    print(f"Allowed local notebook IP for Azure Postgres: {ip}")


def load_postgres_dsn() -> str:
    az = az_command()
    dsn = os.getenv("AZURE_POSTGRES_DSN")
    if dsn:
        return resolve_key_vault_reference(dsn, az)

    query = "[?name=='AZURE_POSTGRES_DSN'].value | [0]"
    command = [
        az,
        "functionapp",
        "config",
        "appsettings",
        "list",
        "--name",
        PROCESSING_FUNCTION_APP,
        "--resource-group",
        RESOURCE_GROUP,
        "--query",
        query,
        "-o",
        "tsv",
    ]
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    dsn = result.stdout.strip()
    if not dsn:
        raise RuntimeError("AZURE_POSTGRES_DSN was not found in the environment or Function App settings.")
    return resolve_key_vault_reference(dsn, az)


if ADD_LOCAL_FIREWALL_RULE:
    allow_current_ip_for_postgres()


dsn = load_postgres_dsn()
print("Loaded Postgres DSN into memory. Value intentionally hidden.")

Loaded Postgres DSN into memory. Value intentionally hidden.


In [5]:
def connect_postgres():
    try:
        return psycopg.connect(
            dsn,
            row_factory=dict_row,
            connect_timeout=CONNECT_TIMEOUT_SECONDS,
        )
    except psycopg.OperationalError as exc:
        message = str(exc).lower()
        if "timeout" in message or "timed out" in message:
            raise TimeoutError(
                "Postgres connection timed out. The DSN resolved correctly, but this machine is probably "
                "not allowed by the Azure Postgres firewall. Set ADD_LOCAL_FIREWALL_RULE = True in the "
                "setup cell and rerun it, or add your current public IP to the Azure Postgres firewall "
                f"for server {POSTGRES_SERVER}."
            ) from exc
        raise


def read_sql(sql: str, params: tuple | dict | None = None) -> pd.DataFrame:
    with connect_postgres() as conn:
        rows = conn.execute(sql, params or ()).fetchall()
    return pd.DataFrame([dict(row) for row in rows])


def scalar_sql(sql: str, params: tuple | dict | None = None):
    with connect_postgres() as conn:
        row = conn.execute(sql, params or ()).fetchone()
    return next(iter(row.values())) if row else None

## Current Data Coverage

This section checks how much processed data is available inside the 90-day summary and connection window.

In [7]:
coverage_sql = """
select
    ie.ticker,
    ie.source,
    count(*)::int as enriched_items,
    count(*) filter (where emb.source_item_id is not null)::int as embedded_items,
    min(si.published_at) as oldest_published_at,
    max(si.published_at) as newest_published_at,
    round(avg(ie.relevance)::numeric, 2) as avg_relevance,
    max(ie.relevance)::int as max_relevance,
    count(*) filter (where ie.firsthand)::int as firsthand_items
from item_enrichments ie
join source_items si on si.source_item_id = ie.source_item_id
left join item_embeddings emb on emb.source_item_id = ie.source_item_id
where ie.ticker = any(%s)
  and ie.source = any(%s)
  and (si.published_at is null or si.published_at >= now() - %s * interval '1 day')
group by ie.ticker, ie.source
order by ie.ticker, ie.source
"""

coverage = read_sql(coverage_sql, (TICKERS, SOURCES, LOOKBACK_DAYS))
coverage

ConnectionTimeout: connection timeout expired

## Summary Threshold Sweep

A summary needs enough relevant items per ticker and enough source diversity to avoid overfitting to one feed. This sweep estimates the summary context that would be available for each `relevance_threshold` and `initial_context_per_source` pair.

In [ ]:
@dataclass(frozen=True)
class SummaryThreshold:
    relevance_threshold: int
    initial_context_per_source: int


summary_grid = [
    SummaryThreshold(relevance_threshold=r, initial_context_per_source=n)
    for r in [0, 1, 2, 3, 4, 5]
    for n in [3, 5, 10, 15]
]


summary_context_sql = """
with ranked as (
    select
        ie.ticker,
        ie.source,
        ie.source_item_id,
        ie.relevance,
        ie.sentiment,
        ie.firsthand,
        si.published_at,
        row_number() over (
            partition by ie.ticker, ie.source
            order by ie.relevance desc, si.published_at desc nulls last
        ) as source_rank
    from item_enrichments ie
    join source_items si on si.source_item_id = ie.source_item_id
    where ie.ticker = any(%s)
      and ie.source = any(%s)
      and ie.relevance >= %s
      and (si.published_at is null or si.published_at >= now() - %s * interval '1 day')
)
select *
from ranked
where source_rank <= %s
"""


def evaluate_summary_threshold(threshold: SummaryThreshold) -> dict:
    rows = read_sql(
        summary_context_sql,
        (TICKERS, SOURCES, threshold.relevance_threshold, LOOKBACK_DAYS, threshold.initial_context_per_source),
    )
    if rows.empty:
        return {
            "relevance_threshold": threshold.relevance_threshold,
            "initial_context_per_source": threshold.initial_context_per_source,
            "tickers_with_items": 0,
            "min_items_per_ticker": 0,
            "median_items_per_ticker": 0,
            "min_sources_per_ticker": 0,
            "avg_relevance": 0,
            "firsthand_items": 0,
            "total_items": 0,
            "summary_score": 0,
        }

    by_ticker = rows.groupby("ticker").agg(
        item_count=("source_item_id", "count"),
        source_count=("source", "nunique"),
        avg_relevance=("relevance", "mean"),
        firsthand_count=("firsthand", "sum"),
    )
    tickers_with_items = by_ticker.shape[0]
    min_items = int(by_ticker["item_count"].min())
    median_items = float(by_ticker["item_count"].median())
    min_sources = int(by_ticker["source_count"].min())
    avg_relevance = float(rows["relevance"].mean())
    firsthand_items = int(rows["firsthand"].sum())
    total_items = int(rows.shape[0])

    coverage_score = tickers_with_items / len(TICKERS)
    depth_score = min(1.0, min_items / 6)
    diversity_score = min(1.0, min_sources / 2)
    quality_score = min(1.0, avg_relevance / 6)
    firsthand_score = min(1.0, firsthand_items / max(1, len(TICKERS)))
    volume_penalty = max(0, (total_items - 120) / 240)
    summary_score = (
        0.35 * coverage_score
        + 0.25 * depth_score
        + 0.20 * diversity_score
        + 0.15 * quality_score
        + 0.05 * firsthand_score
        - 0.10 * volume_penalty
    )

    return {
        "relevance_threshold": threshold.relevance_threshold,
        "initial_context_per_source": threshold.initial_context_per_source,
        "tickers_with_items": tickers_with_items,
        "min_items_per_ticker": min_items,
        "median_items_per_ticker": round(median_items, 1),
        "min_sources_per_ticker": min_sources,
        "avg_relevance": round(avg_relevance, 2),
        "firsthand_items": firsthand_items,
        "total_items": total_items,
        "summary_score": round(summary_score, 3),
    }


summary_results = pd.DataFrame(evaluate_summary_threshold(t) for t in summary_grid)
summary_results.sort_values("summary_score", ascending=False).head(12)

In [ ]:
summary_pivot = summary_results.pivot_table(
    index="relevance_threshold",
    columns="initial_context_per_source",
    values="summary_score",
)
summary_pivot

## Connection Threshold Sweep

Connections require two items from different sources for the same ticker. This sweep computes all cross-source embedding pairs inside the 90-day window, then tests similarity and candidate caps. It also checks how many stored valid connections would pass a given confidence threshold.

The current deterministic connection verifier is intentionally broad, so the most important practical levers are similarity threshold and candidate cap.

In [ ]:
all_pairs_sql = """
select
    least(a.source_item_id, b.source_item_id) as item_a_id,
    greatest(a.source_item_id, b.source_item_id) as item_b_id,
    a.ticker,
    case when a.source_item_id <= b.source_item_id then a.source else b.source end as source_a,
    case when a.source_item_id <= b.source_item_id then b.source else a.source end as source_b,
    case when a.source_item_id <= b.source_item_id then a.published_at else b.published_at end as published_a,
    case when a.source_item_id <= b.source_item_id then b.published_at else a.published_at end as published_b,
    1 - (a.embedding <=> b.embedding) as similarity,
    ia.relevance as relevance_a,
    ib.relevance as relevance_b,
    ia.firsthand as firsthand_a,
    ib.firsthand as firsthand_b,
    a.summary as summary_a,
    b.summary as summary_b
from item_embeddings a
join item_embeddings b
  on a.ticker = b.ticker
 and a.source_item_id < b.source_item_id
 and a.source <> b.source
join item_enrichments ia on ia.source_item_id = a.source_item_id
join item_enrichments ib on ib.source_item_id = b.source_item_id
where a.ticker = any(%s)
  and (a.published_at is null or a.published_at >= now() - %s * interval '1 day')
  and (b.published_at is null or b.published_at >= now() - %s * interval '1 day')
  and (
      a.published_at is null or b.published_at is null or
      abs(extract(epoch from (a.published_at - b.published_at))) <= %s * 24 * 60 * 60
  )
"""

pairs = read_sql(all_pairs_sql, (TICKERS, LOOKBACK_DAYS, LOOKBACK_DAYS, LOOKBACK_DAYS))
pairs["similarity"] = pairs["similarity"].astype(float)
pairs["pair_sources"] = pairs.apply(lambda row: " + ".join(sorted([row["source_a"], row["source_b"]])), axis=1)
pairs["avg_pair_relevance"] = (pairs["relevance_a"] + pairs["relevance_b"]) / 2
pairs.sort_values(["ticker", "similarity"], ascending=[True, False]).head(20)

In [ ]:
pair_distribution = pairs.groupby("ticker").agg(
    pair_count=("item_a_id", "count"),
    source_pair_count=("pair_sources", "nunique"),
    p50_similarity=("similarity", "median"),
    p75_similarity=("similarity", lambda s: s.quantile(0.75)),
    p90_similarity=("similarity", lambda s: s.quantile(0.90)),
    max_similarity=("similarity", "max"),
    avg_pair_relevance=("avg_pair_relevance", "mean"),
).round(3)
pair_distribution

In [ ]:
@dataclass(frozen=True)
class ConnectionThreshold:
    similarity_threshold: float
    confidence_threshold: float
    max_candidates_per_ticker: int


connection_grid = [
    ConnectionThreshold(similarity_threshold=s, confidence_threshold=c, max_candidates_per_ticker=n)
    for s in [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
    for c in [0.25, 0.40, 0.55, 0.70]
    for n in [3, 5, 10, 15]
]


stored_connections_sql = """
select
    ic.ticker,
    ic.item_a_id,
    ic.item_b_id,
    ic.source_a,
    ic.source_b,
    ic.similarity,
    ic.confidence,
    ic.valid,
    ic.narrative,
    ic.stock_relevance,
    ic.verified_at
from item_connections ic
join item_embeddings ea on ea.source_item_id = ic.item_a_id
join item_embeddings eb on eb.source_item_id = ic.item_b_id
where ic.ticker = any(%s)
  and ic.valid = true
  and (ea.published_at is null or ea.published_at >= now() - %s * interval '1 day')
  and (eb.published_at is null or eb.published_at >= now() - %s * interval '1 day')
"""

stored_connections = read_sql(stored_connections_sql, (TICKERS, LOOKBACK_DAYS, LOOKBACK_DAYS))
if not stored_connections.empty:
    stored_connections["similarity"] = stored_connections["similarity"].astype(float)
    stored_connections["confidence"] = stored_connections["confidence"].astype(float)
stored_connections.sort_values(["ticker", "confidence", "similarity"], ascending=[True, False, False]).head(20)

In [ ]:
def evaluate_connection_threshold(threshold: ConnectionThreshold) -> dict:
    eligible = pairs[pairs["similarity"] >= threshold.similarity_threshold].copy()
    if eligible.empty:
        selected = eligible
    else:
        selected = (
            eligible.sort_values(["ticker", "similarity", "avg_pair_relevance"], ascending=[True, False, False])
            .groupby("ticker", group_keys=False)
            .head(threshold.max_candidates_per_ticker)
        )

    stored = stored_connections.copy()
    if not stored.empty:
        stored = stored[
            (stored["similarity"] >= threshold.similarity_threshold)
            & (stored["confidence"] >= threshold.confidence_threshold)
        ]

    if selected.empty:
        tickers_with_candidates = 0
        min_candidates = 0
        median_candidates = 0
        source_pair_count = 0
        avg_similarity = 0
        avg_relevance = 0
        unique_items = 0
    else:
        by_ticker = selected.groupby("ticker").agg(candidate_count=("item_a_id", "count"))
        tickers_with_candidates = by_ticker.shape[0]
        min_candidates = int(by_ticker["candidate_count"].min())
        median_candidates = float(by_ticker["candidate_count"].median())
        source_pair_count = int(selected["pair_sources"].nunique())
        avg_similarity = float(selected["similarity"].mean())
        avg_relevance = float(selected["avg_pair_relevance"].mean())
        unique_items = int(pd.unique(pd.concat([selected["item_a_id"], selected["item_b_id"]])).shape[0])

    total_candidates = int(selected.shape[0])
    stored_valid = int(stored.shape[0]) if not stored.empty else 0
    stored_tickers = int(stored["ticker"].nunique()) if not stored.empty else 0

    coverage_score = tickers_with_candidates / len(TICKERS)
    depth_score = min(1.0, min_candidates / 3)
    source_diversity_score = min(1.0, source_pair_count / 3)
    similarity_score = min(1.0, max(0.0, avg_similarity) / 0.65)
    relevance_score = min(1.0, avg_relevance / 6)
    stored_signal_score = stored_tickers / len(TICKERS)
    volume_penalty = max(0, (total_candidates - len(TICKERS) * threshold.max_candidates_per_ticker) / 50)
    connection_score = (
        0.30 * coverage_score
        + 0.20 * depth_score
        + 0.15 * source_diversity_score
        + 0.15 * similarity_score
        + 0.10 * relevance_score
        + 0.10 * stored_signal_score
        - 0.10 * volume_penalty
    )

    return {
        "similarity_threshold": threshold.similarity_threshold,
        "confidence_threshold": threshold.confidence_threshold,
        "max_candidates_per_ticker": threshold.max_candidates_per_ticker,
        "tickers_with_candidates": tickers_with_candidates,
        "min_candidates_per_ticker": min_candidates,
        "median_candidates_per_ticker": round(median_candidates, 1),
        "source_pair_count": source_pair_count,
        "avg_similarity": round(avg_similarity, 3),
        "avg_pair_relevance": round(avg_relevance, 2),
        "unique_items_connected": unique_items,
        "total_candidates": total_candidates,
        "stored_valid_connections_passing": stored_valid,
        "stored_valid_tickers_passing": stored_tickers,
        "connection_score": round(connection_score, 3),
    }


connection_results = pd.DataFrame(evaluate_connection_threshold(t) for t in connection_grid)
connection_results.sort_values("connection_score", ascending=False).head(20)

In [ ]:
connection_pivot = connection_results.query("max_candidates_per_ticker == 5 and confidence_threshold == 0.25").pivot_table(
    index="similarity_threshold",
    values=["connection_score", "tickers_with_candidates", "total_candidates", "avg_similarity"],
)
connection_pivot

## Inspect Recommended Rows

The tables below select the best-scoring thresholds, then show the exact items or pairs those settings would surface. Use this section to sanity-check whether the recommendation is actually useful, not just numerically convenient.

In [ ]:
best_summary = summary_results.sort_values("summary_score", ascending=False).iloc[0].to_dict()
best_connection = connection_results.sort_values("connection_score", ascending=False).iloc[0].to_dict()

best_summary, best_connection

In [ ]:
recommended_summary_items = read_sql(
    summary_context_sql,
    (
        TICKERS,
        SOURCES,
        int(best_summary["relevance_threshold"]),
        LOOKBACK_DAYS,
        int(best_summary["initial_context_per_source"]),
    ),
)

recommended_summary_items.groupby(["ticker", "source"]).agg(
    items=("source_item_id", "count"),
    avg_relevance=("relevance", "mean"),
    newest_published_at=("published_at", "max"),
).round(2)

In [ ]:
recommended_pairs = pairs[pairs["similarity"] >= float(best_connection["similarity_threshold"])].copy()
recommended_pairs = (
    recommended_pairs.sort_values(["ticker", "similarity", "avg_pair_relevance"], ascending=[True, False, False])
    .groupby("ticker", group_keys=False)
    .head(int(best_connection["max_candidates_per_ticker"]))
)

recommended_pairs[[
    "ticker",
    "source_a",
    "source_b",
    "similarity",
    "avg_pair_relevance",
    "item_a_id",
    "item_b_id",
    "summary_a",
    "summary_b",
]].sort_values(["ticker", "similarity"], ascending=[True, False]).head(50)

## Recommendation Template

Run the cells above, then use the generated best rows and item inspection to decide whether to keep the current broad settings or tighten them. A good threshold set should satisfy these checks:

- Every ticker has summary items in the 90-day window.
- Most tickers have at least two sources represented in summary context.
- Connection candidates exist for as many tickers as the data allows.
- The selected connection pairs are readable business leads, not just lexical collisions.
- `max_connection_candidates_per_ticker` keeps review volume manageable.

Current working hypothesis before manual inspection: keep summary relevance broad, keep confidence at `0.25`, and use `max_connection_candidates_per_ticker` as the primary volume control. Raise `similarity_threshold` only if the recommended-pair inspection shows too many weak semantic links.

In [ ]:
recommendation = {
    "lookback_days": LOOKBACK_DAYS,
    "summary": {
        "relevance_threshold": int(best_summary["relevance_threshold"]),
        "initial_context_per_source": int(best_summary["initial_context_per_source"]),
        "score": float(best_summary["summary_score"]),
        "min_items_per_ticker": int(best_summary["min_items_per_ticker"]),
        "min_sources_per_ticker": int(best_summary["min_sources_per_ticker"]),
    },
    "connections": {
        "similarity_threshold": float(best_connection["similarity_threshold"]),
        "connection_confidence_threshold": float(best_connection["confidence_threshold"]),
        "max_connection_candidates_per_ticker": int(best_connection["max_candidates_per_ticker"]),
        "score": float(best_connection["connection_score"]),
        "tickers_with_candidates": int(best_connection["tickers_with_candidates"]),
        "total_candidates": int(best_connection["total_candidates"]),
    },
}

print(json.dumps(recommendation, indent=2))